# Titanic dataset
Based on Ensemble models like RFC or XGBoost. Надо сравнить метрики их качества и узнать насколько они отличаются

In [9]:
# Импорт всех библиотек

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [10]:
# Загрузка train данных

train_data = pd.read_csv("titanic_dataset/train.csv")
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [11]:
# Загрузка test данных
test_data = pd.read_csv("titanic_dataset//test.csv")
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [12]:
women = train_data.loc[train_data["Sex"] == "female"]["Survived"]
print(f'Процент выживших женщин = {sum(women) / len(women)}')

women = train_data.loc[train_data["Sex"] == "male"]["Survived"]
print(f'Процент выживших мужчин = {sum(women) / len(women)}')

Процент выживших женщин = 0.7420382165605095
Процент выживших мужчин = 0.18890814558058924


In [13]:
# Data for RFC

y = train_data["Survived"]

features = ["Pclass", "Sex", "SibSp", "Parch"]

X = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])

# RFC train + CV

param_dist_rfc = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 5, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None]
}

rfc = RandomForestClassifier()
rfc_cv = RandomizedSearchCV(
    rfc,
    param_dist_rfc,
    n_iter=30,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

rfc_cv.fit(X, y)

print(f'Лучшая модель: {rfc_cv.best_estimator_}\nЛучшие гиперпараметры: {rfc_cv.best_params_},\nЛучшая метрика: {rfc_cv.best_score_}')

Лучшая модель: RandomForestClassifier(max_depth=5, min_samples_split=5, n_estimators=500)
Лучшие гиперпараметры: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 5},
Лучшая метрика: 0.8013872324398971


In [14]:
# Test predictions + output RFC

rfc_best = rfc_cv.best_estimator_

y_test = rfc_best.predict(X_test)
output = pd.DataFrame({
    "PassengerId": test_data.PassengerId,
    "Survived": y_test
})

output.to_csv("submission.csv", index=False)

In [15]:
# XGBoost train CV

xgb = XGBClassifier()

param_dist_xgb = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 7, 10],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0, 0.1, 0.3]
}

xgb_cv = RandomizedSearchCV(
    xgb,
    param_distributions=param_dist_xgb,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    verbose=1,
    n_jobs=-1
)

xgb_cv.fit(X, y)
print(f'Лучшая модель: {xgb_cv.best_estimator_}\nЛучшие гиперпараметры: {xgb_cv.best_params_},\nЛучшая метрика: {xgb_cv.best_score_}')

xgb = XGBClassifier()
xgb.fit(X, y)
print(f'Лучшая модель: {xgb}\nЛучшие гиперпараметры: {xgb.get_params()},\nЛучшая метрика: {xgb.score(X, y)}')

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Лучшая модель: XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=1.0, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=0.1, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)
Лучшие гиперпараметры: {'subsample': 1.0, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 1.0},
Лучшая метрика: 0.8036093151

In [16]:
# CatBoost train CV

cat_features = ["Sex", "Embarked"]

cat = CatBoostClassifier(
    iterations=200,
    learning_rate=0.1,
    depth=6,
    verbose=0)

param_dist_cat = {
    "iterations": [100, 200, 300],
    "depth": [4, 6, 8, 10],
    "learning_rate": [0.01, 0.05, 0.1],
    "l2_leaf_reg": [1, 3, 5, 7]
}

cat_cv = RandomizedSearchCV(
    cat,
    param_distributions=param_dist_cat)

cat_cv.fit(X, y)

print(f'Лучшая модель: {cat_cv.best_estimator_}\nЛучшие гиперпараметры: {cat_cv.best_params_},\nЛучшая метрика: {cat_cv.best_score_}')

Лучшая модель: CatBoostClassifier(depth=4, iterations=100, l2_leaf_reg=7, learning_rate=0.1, verbose=0)
Лучшие гиперпараметры: {'learning_rate': 0.1, 'l2_leaf_reg': 7, 'iterations': 100, 'depth': 4},
Лучшая метрика: 0.8069549934090766


In [19]:
# Test predictions + output Cat

cat = CatBoostClassifier(
    iterations=200,
    learning_rate=0.1,
    depth=6,
    verbose=0)

cat.fit(X, y)
best_cat = cat_cv.best_estimator_

y_test = best_cat.predict(X_test)
output = pd.DataFrame({
    "PassengerId": test_data.PassengerId,
    "Survived": y_test
})

# output.to_csv("submission.csv", index=False)

In [18]:
# Save cat model

cat.save_model("cat_model.cbm")